# Campaña Avícola — EDA y Limpieza de Datos (Notebook 1/2)

Exploración, tratamiento de outliers, ingeniería de características y análisis estadístico del dataset
de ventas por campaña avícola. La salida es un dataset limpio que alimenta el notebook de modelado
(`02_modelo_pycaret.ipynb`).

**Pipeline:** carga → EDA → tipado → outliers (IQR) → feature engineering → correlación/normalidad/multicolinealidad → baseline lineal → exportación.

> Coloca `Campaña Prueba.xlsx` dentro de `data/` (no se versiona, ver `.gitignore`).

## 1. Configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

SEED = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Carga y exploración inicial (EDA)

In [ ]:
df = pd.read_excel("data/Campaña Prueba.xlsx")
print(f"Registros: {df.shape[0]} | Variables: {df.shape[1]}")
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
# Identificadores que deben tratarse como categoricos, no numericos
df["Centro"] = df["Centro"].astype("object")
df["Código de Campaña"] = df["Código de Campaña"].astype("object")

# Distribucion de la variable categorica relevante
df["Zona"].value_counts(normalize=True).round(2).plot.barh()
plt.title("Distribucion por Zona"); plt.tight_layout(); plt.show()

## 3. Resumen y valores nulos

In [ ]:
def resumen_variables(df: pd.DataFrame) -> pd.DataFrame:
    """Resumen por columna: moda, frecuencia de la moda, desviacion y coeficiente de variacion."""
    filas = []
    for col in df.columns:
        moda = df[col].mode().iloc[0] if not df[col].mode().empty else None
        frec = df[col].value_counts().iloc[0] if moda is not None else None
        if pd.api.types.is_numeric_dtype(df[col]):
            std = df[col].std()
            media = df[col].mean()
            cv = (std / media * 100) if media else None
        else:
            std = cv = None
        filas.append({"columna": col, "moda": moda, "frec_moda": frec,
                      "std": std, "cv_%": cv})
    return pd.DataFrame(filas)

resumen_variables(df)

In [ ]:
print("Valores nulos por columna:")
print(df.isnull().sum())

## 4. Detección y tratamiento de outliers (IQR)

In [ ]:
def contar_outliers_iqr(df: pd.DataFrame) -> pd.DataFrame:
    """Cuenta outliers por columna numerica usando el rango intercuartilico (1.5*IQR)."""
    filas = []
    for col in df.select_dtypes(include=["int64", "float64"]).columns:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        out = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
        filas.append({"columna": col, "outliers": out})
    return pd.DataFrame(filas)

def recortar_outliers(df: pd.DataFrame) -> pd.DataFrame:
    """Recorta (clipping) los valores fuera de 1.5*IQR a los limites, en vez de eliminarlos."""
    df = df.copy()
    for col in df.select_dtypes(include=["int64", "float64"]).columns:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        df[col] = df[col].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)
    return df

print("Outliers antes del tratamiento:")
print(contar_outliers_iqr(df).to_string(index=False))

In [ ]:
df = recortar_outliers(df)
print("Outliers despues del recorte (clipping):")
print(contar_outliers_iqr(df).to_string(index=False))

## 5. Ingeniería de características

In [ ]:
# La columna Proveedor tiene demasiadas categorias; se extrae la razon social base
# (p. ej. "REYNA 2" -> "REYNA") para reducir la cardinalidad.
df["ProveedorRS"] = df["Proveedor"].str.extract(r"([A-Za-záéíóúÑñ\s]+)")[0].str.strip()
df = df.drop(columns=["Proveedor"])
df["ProveedorRS"].value_counts(normalize=True).round(3).head(10)

In [ ]:
# Se descartan columnas sin valor para el analisis (identificador y constantes)
df = df.drop(columns=["Centro", "Cojo", "Nro"])
df.info()

## 6. Correlación, normalidad y multicolinealidad

In [ ]:
num_cols = ["Primera", "Brasa", "Segunda", "Descarte",
            "Venta en Unidades", "Venta en Kilogramos", "Peso en Ventas", "Edad en Ventas"]

plt.figure(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Matriz de correlacion"); plt.tight_layout(); plt.show()

In [ ]:
# Normalidad (Shapiro-Wilk): p<0.05 -> no normal
print("Prueba de normalidad (Shapiro-Wilk):")
for col in num_cols:
    _, p = shapiro(df[col].dropna())
    print(f"  {col}: p={p:.4f} -> {'no normal' if p < 0.05 else 'normal'}")

In [ ]:
# Multicolinealidad (VIF): valores altos indican variables redundantes
X_vif = sm.add_constant(df[num_cols])
vif = pd.DataFrame({"Variable": X_vif.columns,
                    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]})
print(vif.to_string(index=False))

## 7. Baseline: regresión lineal

Se ajusta una regresión lineal simple como línea base para predecir `Primera`. Su desempeño ayuda a
decidir si conviene un modelo no lineal (Random Forest / Gradient Boosting), que se explora en el
notebook de modelado.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

features = ["Venta en Unidades", "Venta en Kilogramos", "Peso en Ventas",
            "Edad en Ventas", "Brasa", "Segunda", "Descarte"]
X, y = df[features], df["Primera"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)

lr = LinearRegression().fit(X_tr, y_tr)
y_pred = lr.predict(X_te)
print(f"MAE  : {mean_absolute_error(y_te, y_pred):,.0f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_te, y_pred)):,.0f}")
print(f"R2   : {r2_score(y_te, y_pred):.3f}")
mape = np.mean(np.abs((y_te - y_pred) / y_te)) * 100
print(f"MAPE : {mape:.2f}%")

## 8. Exportación del dataset limpio

In [ ]:
df.to_excel("data/campania_procesado.xlsx", index=False)
print("Dataset limpio guardado en data/campania_procesado.xlsx")
df.head()